In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/naghamalshbrawi/hybrid-arat5v2/final_df_with_hybrid_AraT5v2_simplifications.csv
/kaggle/input/datasets/naghamalshbrawi/arabart/final_df_with_hybrid_AraBART_simplifications.csv


In [2]:
!pip install -q sacrebleu bert-score evaluate rouge-score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [3]:
!pip install -q sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 34.1 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np

import sacrebleu
import evaluate
import nltk

from bert_score import score as bert_score

print("All imports loaded successfully")
print("sacrebleu version:", sacrebleu.__version__)

All imports loaded successfully
sacrebleu version: 2.6.0


In [5]:
predictions = ["هذا نص مبسط"]
references = ["هذا نص بسيط"]

bleu = sacrebleu.corpus_bleu(predictions, [references])

print("BLEU works")
print("BLEU score:", bleu.score)

BLEU works
BLEU score: 0.0


In [6]:
P, R, F1 = bert_score(
    predictions,
    references,
    lang="ar",
    verbose=False
)

print("BERTScore works")
print("BERTScore F1:", F1.mean().item())

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore works
BERTScore F1: 0.9151749610900879


In [7]:
sari_metric = evaluate.load("sari")

sources = ["هذا نص صعب جدا"]
predictions = ["هذا نص بسيط"]
references = [["هذا نص سهل"]]

sari = sari_metric.compute(
    sources=sources,
    predictions=predictions,
    references=references
)

print("SARI works")
print(sari)

SARI works
{'sari': 75.0}


In [ ]:
import pandas as pd
import numpy as np
import sacrebleu
import evaluate
from bert_score import score as bert_score
from nltk.translate.meteor_score import meteor_score
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

sari_metric = evaluate.load("sari")

print("Imports ready")

In [9]:
file_path = "/kaggle/input/datasets/naghamalshbrawi/arabart/final_df_with_hybrid_AraBART_simplifications.csv"   

df = pd.read_csv(file_path)

required_cols = [
    "Original_Sentence",
    "ref_level_3", "ref_level_2", "ref_level_1",
    "hybrid_level_3", "hybrid_level_2", "hybrid_level_1"
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")
else:
    print("✅ All required columns exist")

df = df[required_cols].fillna("").astype(str)
df.head()

✅ All required columns exist


,Original_Sentence,ref_level_3,ref_level_2,ref_level_1,hybrid_level_3,hybrid_level_2,hybrid_level_1
0,"•• ""عدم الانحياز"" اصطلاح سياسي يعنى الحياد..","""عدم الانحياز"" هو مصطلح سياسي يشير إلى الحياد.","""عدم الانحياز"" هو مصطلح سياسي يعني أن تكون محا...","""عدم الانحياز"" يعني أن تكون محايداً ولا تفضل أ...","""عدم الرضا"" مصطلح سياسي يعني الاستقلال.","""عدم الرضا"" هو مصطلح عربي يعني الاستقلال.","""عدم الرضا"" هو مصطلح عربي يعني الاستقلال."
1,حيث تلتزم الدول غير المنحازة بعدم الارتباط بأي...,تلتزم الدول غير المنحازة بالحياد وعدم التحالف ...,تلتزم الدول غير المنحازة بعدم التحالف مع دول ا...,الدول غير المنحازة لا تتعامل مع دول الشرق أو ا...,تلتزم الدول غير المنحازة بعدم الارتباط بأي دول...,تلتزم الدول غير المنحازة بعدم الارتباط بأي دول...,الدول غير المنظمة لا تتعامل مع دول أخرى في الش...
2,بمعنى أن الدول غير المنحازة لا تقف موقف المتفر...,بمعنى أن الدول غير المنحازة ليس موقفها موقف مت...,الدول غير المنحازة ليس لها موقف محايد في القضا...,الدول غير المنحازة ليسوا متفرجين، هم يتدخلون و...,بمعنى أن الدول غير العظمى لا تقف موقف واحد أما...,الدول غير المنظمة لا تقف موقف واحد أمام المشاك...,الدول غير المنظمة لا تقف متفرجة أمام المشاكل ا...
3,وقد بدأ استعمال اصطلاح (عدم الانحياز) في مؤتمر...,"وقد بدأ استعمال اصطلاح ""عدم الانحياز"" في مؤتمر...","وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استخدام مصطلح ""عدم التمييز"" في مؤتمر ب...","وقد بدأ استخدام مصطلح ""عدم التمييز"" في مؤتمر ب...","وقد بدأ استخدام مصطلح ""عدم المشاركة"" في مؤتمر ..."
4,••السيارات التي يستعملها الدبلوماسيون.. أي رجا...,تتحمل السيارات الدبلوماسية، أي تلك التي يستخدم...,تتحمل السيارات التي يستخدمها الدبلوماسيون جملة...,"السيارات التي يستخدمها الدبلوماسيون تسمى ""هيئة...",ال سيارات التي يستعملها الدبلوماسيون، أي رجال ...,"يستخدم الدبلوماسيون هذه الجملة لوصف ""هيئة سياس...",الدبلوماسيون هم أشخاص دبلوماسيون معتمدون لدى ا...


In [10]:
def compute_sari(sources, predictions, references):
    """
    SARI needs:
    sources = original sentences
    predictions = model outputs
    references = list of references
    """
    refs = [[ref] for ref in references]

    result = sari_metric.compute(
        sources=sources,
        predictions=predictions,
        references=refs
    )
    return result["sari"]


def compute_bleu(predictions, references):
    """
    BLEU between prediction and reference.
    """
    return sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score


def compute_fbertscore(predictions, references):
    """
    BERTScore F1.
    """
    P, R, F1 = bert_score(
        predictions,
        references,
        lang="ar",
        verbose=False
    )
    return F1.mean().item() * 100


def compute_osman_proxy(predictions, references):
    """
    OSMAN proxy using METEOR-style similarity.
    Higher = closer to reference.
    """
    scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = pred.split()
        ref_tokens = ref.split()

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            scores.append(0)
        else:
            scores.append(meteor_score([ref_tokens], pred_tokens) * 100)

    return np.mean(scores)

In [11]:
# Evaluate Hybrid Model per level

sources = df["Original_Sentence"].tolist()

levels = {
    "Level 1": {
        "reference": df["ref_level_1"].tolist(),
        "prediction": df["hybrid_level_1"].tolist()
    },
    "Level 2": {
        "reference": df["ref_level_2"].tolist(),
        "prediction": df["hybrid_level_2"].tolist()
    },
    "Level 3": {
        "reference": df["ref_level_3"].tolist(),
        "prediction": df["hybrid_level_3"].tolist()
    }
}

results = []

for level_name, data in levels.items():
    refs = data["reference"]
    preds = data["prediction"]

    sari = compute_sari(sources, preds, refs)
    bleu_ref = compute_bleu(preds, refs)
    bleu_input = compute_bleu(preds, sources)
    fbertscore = compute_fbertscore(preds, refs)
    osman = compute_osman_proxy(preds, refs)

    results.append({
        "Model": "Hybrid",
        "Level": level_name,
        "SARI": sari,
        "BLEU_pred_ref": bleu_ref,
        "BLEU_pred_input": bleu_input,
        "FBERTScore": fbertscore,
        "OSMAN_proxy": osman
    })

results_df = pd.DataFrame(results)
results_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Model,Level,SARI,BLEU_pred_ref,BLEU_pred_input,FBERTScore,OSMAN_proxy
0,Hybrid,Level 1,52.403637,14.997387,22.267608,81.632829,31.080199
1,Hybrid,Level 2,52.547420,24.669511,36.853523,84.793758,42.686844
2,Hybrid,Level 3,51.622817,27.811000,42.892346,85.845500,47.167582


In [12]:
# Save results
output_path = "/content/hybrid_AraBART_results.csv"

results_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Results saved to:", output_path)
results_df

Results saved to: /content/hybrid_AraBART_results.csv


,Model,Level,SARI,BLEU_pred_ref,BLEU_pred_input,FBERTScore,OSMAN_proxy
0,Hybrid,Level 1,52.403637,14.997387,22.267608,81.632829,31.080199
1,Hybrid,Level 2,52.547420,24.669511,36.853523,84.793758,42.686844
2,Hybrid,Level 3,51.622817,27.811000,42.892346,85.845500,47.167582


arat5v2

In [13]:
file_path = "/kaggle/input/datasets/naghamalshbrawi/hybrid-arat5v2/final_df_with_hybrid_AraT5v2_simplifications.csv"   

df = pd.read_csv(file_path)

required_cols = [
    "Original_Sentence",
    "ref_level_3", "ref_level_2", "ref_level_1",
    "hybrid_level_3", "hybrid_level_2", "hybrid_level_1"
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")
else:
    print("✅ All required columns exist")

df = df[required_cols].fillna("").astype(str)
df.head()

✅ All required columns exist


,Original_Sentence,ref_level_3,ref_level_2,ref_level_1,hybrid_level_3,hybrid_level_2,hybrid_level_1
0,"•• ""عدم الانحياز"" اصطلاح سياسي يعنى الحياد..","""عدم الانحياز"" هو مصطلح سياسي يشير إلى الحياد.","""عدم الانحياز"" هو مصطلح سياسي يعني أن تكون محا...","""عدم الانحياز"" يعني أن تكون محايداً ولا تفضل أ...","""عدم الرضا"" مصطلح عربي يعني الاستقلال.","""عدم الرضا"" مصطلح عربي يعني الاستقلال.",عدم الاستقرار هو التدخل في الأمور العامة، وهو ...
1,حيث تلتزم الدول غير المنحازة بعدم الارتباط بأي...,تلتزم الدول غير المنحازة بالحياد وعدم التحالف ...,تلتزم الدول غير المنحازة بعدم التحالف مع دول ا...,الدول غير المنحازة لا تتعامل مع دول الشرق أو ا...,تلتزم الدول غير المنحازة بعدم الارتباط بدول ال...,تلتزم الدول غير المنحازة بعدم الارتباط بدول ال...,الدول غير المنظمة لا تتعامل مع دول الشرق أو ال...
2,بمعنى أن الدول غير المنحازة لا تقف موقف المتفر...,بمعنى أن الدول غير المنحازة ليس موقفها موقف مت...,الدول غير المنحازة ليس لها موقف محايد في القضا...,الدول غير المنحازة ليسوا متفرجين، هم يتدخلون و...,الدول غير المنحازة لا تقف موقف واحد أمام المشا...,الدول غير المنحازة لا تقف موقف واحد أمام المشا...,الدول غير المنظمة لا تقف موقف واحد أمام المشاك...
3,وقد بدأ استعمال اصطلاح (عدم الانحياز) في مؤتمر...,"وقد بدأ استعمال اصطلاح ""عدم الانحياز"" في مؤتمر...","وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استخدام مصطلح ""عدم الاستقلال"" في مؤتمر...","بدأ استخدام مصطلح ""عدم الاستقلال"" في مؤتمر بلغ...","بدأ استخدام مصطلح ""عدم الاستقلال"" في مؤتمر بلغ..."
4,••السيارات التي يستعملها الدبلوماسيون.. أي رجا...,تتحمل السيارات الدبلوماسية، أي تلك التي يستخدم...,تتحمل السيارات التي يستخدمها الدبلوماسيون جملة...,"السيارات التي يستخدمها الدبلوماسيون تسمى ""هيئة...",السيارات التي يستخدمها الدبلوماسيون، أي رجال ا...,السيارات التي يستخدمها الدبلوماسيون، أي رجال ا...,"الدبلوماسيون يستخدمون سيارات تحمل كلمة ""هيئة ا..."


In [14]:
def compute_sari(sources, predictions, references):
    """
    SARI needs:
    sources = original sentences
    predictions = model outputs
    references = list of references
    """
    refs = [[ref] for ref in references]

    result = sari_metric.compute(
        sources=sources,
        predictions=predictions,
        references=refs
    )
    return result["sari"]


def compute_bleu(predictions, references):
    """
    BLEU between prediction and reference.
    """
    return sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score


def compute_fbertscore(predictions, references):
    """
    BERTScore F1.
    """
    P, R, F1 = bert_score(
        predictions,
        references,
        lang="ar",
        verbose=False
    )
    return F1.mean().item() * 100


def compute_osman_proxy(predictions, references):
    """
    OSMAN proxy using METEOR-style similarity.
    Higher = closer to reference.
    """
    scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = pred.split()
        ref_tokens = ref.split()

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            scores.append(0)
        else:
            scores.append(meteor_score([ref_tokens], pred_tokens) * 100)

    return np.mean(scores)

In [ ]:
# Evaluate Hybrid Model per level

sources = df["Original_Sentence"].tolist()

levels = {
    "Level 1": {
        "reference": df["ref_level_1"].tolist(),
        "prediction": df["hybrid_level_1"].tolist()
    },
    "Level 2": {
        "reference": df["ref_level_2"].tolist(),
        "prediction": df["hybrid_level_2"].tolist()
    },
    "Level 3": {
        "reference": df["ref_level_3"].tolist(),
        "prediction": df["hybrid_level_3"].tolist()
    }
}

results = []

for level_name, data in levels.items():
    refs = data["reference"]
    preds = data["prediction"]

    sari = compute_sari(sources, preds, refs)
    bleu_ref = compute_bleu(preds, refs)
    bleu_input = compute_bleu(preds, sources)
    fbertscore = compute_fbertscore(preds, refs)
    osman = compute_osman_proxy(preds, refs)

    results.append({
        "Model": "Hybrid",
        "Level": level_name,
        "SARI": sari,
        "BLEU_pred_ref": bleu_ref,
        "BLEU_pred_input": bleu_input,
        "FBERTScore": fbertscore,
        "OSMAN_proxy": osman
    })

results_df = pd.DataFrame(results)
results_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Model,Level,SARI,BLEU_pred_ref,BLEU_pred_input,FBERTScore,OSMAN_proxy
0,Hybrid,Level 1,54.355280,16.593125,19.928895,82.135350,33.796969
1,Hybrid,Level 2,54.805123,26.953580,34.700579,85.377711,45.677787
2,Hybrid,Level 3,53.563180,29.858661,42.695253,86.367285,49.742397


In [ ]:
# Save results
output_path = "/content/hybrid_arat5v2_results.csv"

results_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Results saved to:", output_path)
results_df

Results saved to: /content/hybrid_arat5v2_results.csv


,Model,Level,SARI,BLEU_pred_ref,BLEU_pred_input,FBERTScore,OSMAN_proxy
0,Hybrid,Level 1,54.355280,16.593125,19.928895,82.135350,33.796969
1,Hybrid,Level 2,54.805123,26.953580,34.700579,85.377711,45.677787
2,Hybrid,Level 3,53.563180,29.858661,42.695253,86.367285,49.742397
